<a href="https://colab.research.google.com/github/ynam0327-afk/REDRED/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import joblib
from google.colab import drive
from xgboost import XGBClassifier

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [8]:
from sklearn.model_selection import RandomizedSearchCV

# ==================================
# 1. 데이터 불러오기
# ==================================

df = pd.read_csv("/content/drive/MyDrive/REDRED/url_features.csv")

feature_cols = [
    "url_length", "domain_length", "dot_count", "slash_count", "hyphen_count",
    "digit_count", "contains_at", "has_ip", "https_flag", "shortener",
    "suspicious_word_count",
]

X = df[feature_cols]
y = df["label"]


# ==================================
# 2. Train/Test 분리
# ==================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train :", X_train.shape)
print("Test  :", X_test.shape)

# 불균형 보정용 비율 (XGBoost의 scale_pos_weight에 사용)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", round(scale_pos_weight, 3))


# ==================================
# 3. Random Forest 랜덤서치 (속도 개선)
# ==================================

rf_param_grid = {
    "n_estimators": [100, 150, 200],
    "max_depth": [10, 15, 18, 25],  # None 제거 - 무제한 깊이는 대용량에서 매우 느림
}

rf = RandomForestClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=1,  # 중첩 병렬처리 방지 - 병렬은 RandomizedSearchCV 쪽(n_jobs=-1)에서만
)

rf_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=rf_param_grid,
    n_iter=4,          # 전수 조합(12개) 대신 4개만 랜덤 시도
    scoring="roc_auc",
    cv=2,              # 3-fold -> 2-fold로 축소
    n_jobs=-1,
    random_state=42,
    verbose=2,
)

rf_search.fit(X_train, y_train)

print("\nRandom Forest 최적 파라미터")
print(rf_search.best_params_)

best_rf = rf_search.best_estimator_

rf_pred = best_rf.predict(X_test)
rf_prob = best_rf.predict_proba(X_test)[:, 1]

print("\n========================")
print("Random Forest (튜닝됨)")
print("========================")
print("Accuracy :", round(accuracy_score(y_test, rf_pred), 4))
print("Precision:", round(precision_score(y_test, rf_pred), 4))
print("Recall   :", round(recall_score(y_test, rf_pred), 4))
print("F1-score :", round(f1_score(y_test, rf_pred), 4))
print("ROC-AUC  :", round(roc_auc_score(y_test, rf_prob), 4))
print("\nClassification Report")
print(classification_report(y_test, rf_pred))


# ==================================
# 4. XGBoost 랜덤서치 (불균형 보정 추가, 속도 개선)
# ==================================

param_grid = {
    "n_estimators": [100, 150, 200],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.05, 0.1, 0.2],
}

xgb = XGBClassifier(
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,  # RF의 class_weight="balanced"와 대응
    random_state=42,
    n_jobs=1,  # 중첩 병렬처리 방지
)

xgb_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_grid,
    n_iter=4,
    scoring="roc_auc",
    cv=2,
    n_jobs=-1,
    random_state=42,
    verbose=2,
)

xgb_search.fit(X_train, y_train)

print("\nXGBoost 최적 파라미터")
print(xgb_search.best_params_)

best_xgb = xgb_search.best_estimator_


# ==================================
# 5. XGBoost 평가
# ==================================

xgb_pred = best_xgb.predict(X_test)
xgb_prob = best_xgb.predict_proba(X_test)[:, 1]

print("\n========================")
print("XGBoost (튜닝됨 + 불균형 보정)")
print("========================")
print("Accuracy :", round(accuracy_score(y_test, xgb_pred), 4))
print("Precision:", round(precision_score(y_test, xgb_pred), 4))
print("Recall   :", round(recall_score(y_test, xgb_pred), 4))
print("F1-score :", round(f1_score(y_test, xgb_pred), 4))
print("ROC-AUC  :", round(roc_auc_score(y_test, xgb_prob), 4))
print("\nClassification Report")
print(classification_report(y_test, xgb_pred))


# ==================================
# 6. 중요 변수 확인
# ==================================

rf_importance = pd.Series(
    best_rf.feature_importances_, index=feature_cols
).sort_values(ascending=False)

print("\nRandom Forest Feature Importance")
print(rf_importance)


# ==================================
# 7. 모델 저장
# ==================================

joblib.dump(best_rf, "/content/drive/MyDrive/REDRED/rf_url_model.joblib", compress=3)
joblib.dump(best_xgb, "/content/drive/MyDrive/REDRED/xgb_url_model.joblib", compress=3)

print("\n모델 저장 완료")


Train : (520952, 11)
Test  : (130239, 11)
scale_pos_weight: 1.919
Fitting 2 folds for each of 4 candidates, totalling 8 fits

Random Forest 최적 파라미터
{'n_estimators': 150, 'max_depth': 25}

Random Forest (튜닝됨)
Accuracy : 0.9271
Precision: 0.8764
Recall   : 0.9164
F1-score : 0.896
ROC-AUC  : 0.9788

Classification Report
              precision    recall  f1-score   support

           0       0.96      0.93      0.94     85621
           1       0.88      0.92      0.90     44618

    accuracy                           0.93    130239
   macro avg       0.92      0.92      0.92    130239
weighted avg       0.93      0.93      0.93    130239

Fitting 2 folds for each of 4 candidates, totalling 8 fits


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



XGBoost 최적 파라미터
{'n_estimators': 200, 'max_depth': 8, 'learning_rate': 0.05}

XGBoost (튜닝됨 + 불균형 보정)
Accuracy : 0.8961
Precision: 0.8092
Recall   : 0.9118
F1-score : 0.8574
ROC-AUC  : 0.9642

Classification Report
              precision    recall  f1-score   support

           0       0.95      0.89      0.92     85621
           1       0.81      0.91      0.86     44618

    accuracy                           0.90    130239
   macro avg       0.88      0.90      0.89    130239
weighted avg       0.90      0.90      0.90    130239


Random Forest Feature Importance
slash_count              0.275125
dot_count                0.202068
url_length               0.170770
domain_length            0.134028
digit_count              0.094172
hyphen_count             0.051650
https_flag               0.026927
suspicious_word_count    0.022943
shortener                0.011515
has_ip                   0.009989
contains_at              0.000813
dtype: float64

모델 저장 완료


**Scoring**

In [3]:
import re
import joblib
import numpy as np
import pandas as pd
from urllib.parse import urlparse

FEATURE_COLS = [
    "url_length", "domain_length", "dot_count", "slash_count", "hyphen_count",
    "digit_count", "contains_at", "has_ip", "https_flag", "shortener",
    "suspicious_word_count",
]

# 2주차 도메인 구조 분석 모듈(domain_analyzer.py)과 동일한 단축 URL 목록.
# 두 모듈이 서로 다른 리스트를 쓰면 shortener 피처가 어긋나므로 반드시 동기화 유지.
SHORTENER_DOMAINS = {
    "vo.la", "url.kr", "bit.ly", "han.gl", "t.ly", "goo.gl", "tinyurl.com",
    "t.co", "ow.ly", "is.gd", "buff.ly", "cutt.ly", "rb.gy",
}

SUSPICIOUS_WORDS = ["login", "verify", "secure", "account", "update", "confirm", "banking"]


def extract_url_features(raw_url: str) -> dict:
    """
    URL 문자열 하나에서 학습 때 쓰인 11개 피처를 동일한 방식으로 추출한다.
    """
    url = str(raw_url).strip()
    if not re.match(r"^https?://", url):
        url = "http://" + url

    parsed = urlparse(url)
    domain = parsed.netloc.lower()

    return {
        "url_length": len(url),
        "domain_length": len(domain),
        "dot_count": url.count("."),
        "slash_count": url.count("/"),
        "hyphen_count": url.count("-"),
        "digit_count": sum(c.isdigit() for c in url),
        "contains_at": int("@" in url),
        "has_ip": int(bool(re.match(r"^\d{1,3}(\.\d{1,3}){3}$", domain))),
        "https_flag": int(url.startswith("https")),
        "shortener": int(domain in SHORTENER_DOMAINS),
        "suspicious_word_count": sum(w in url.lower() for w in SUSPICIOUS_WORDS),
    }


def load_url_model(path: str = "/content/drive/MyDrive/REDRED/rf_url_model.joblib"):
    return joblib.load(path)


# 2주차 domain_analyzer.py와 동일한 화이트리스트 - 모델보다 먼저 확인한다.
OFFICIAL_DOMAINS = {
    "safekorea.go.kr", "korea119.go.kr", "nfa.go.kr", "weather.go.kr", "kma.go.kr",
    "vo.la", "url.kr",
}
OFFICIAL_TLD_SUFFIXES = (".go.kr",)


def is_official_domain(domain: str) -> bool:
    if any(domain == d or domain.endswith("." + d) for d in OFFICIAL_DOMAINS):
        return True
    return any(domain.endswith(suffix) for suffix in OFFICIAL_TLD_SUFFIXES)


def url_risk_score_model(raw_url: str, model) -> float:

    url = str(raw_url).strip()
    if not re.match(r"^https?://", url):
        url = "http://" + url
    domain = urlparse(url).netloc.lower()

    if is_official_domain(domain):
        return 0.0

    features = extract_url_features(raw_url)
    X = pd.DataFrame([features])[FEATURE_COLS]
    proba = model.predict_proba(X)[0][1]
    return float(proba)


# ---------------------------------------------------------------------------
# 2주차 disaster_reliability_score, final_alert_score는 그대로 유지
# ---------------------------------------------------------------------------
def disaster_reliability_score(match_info: dict) -> dict:

    if not match_info.get("region_match"):
        return {
            "score": 0.5,
            "note": "정보 부족 - 관할 시군구에 대응하는 소방청 사건 없음 (판단보류, 위험 아님)",
        }

    # 시군구가 실제로 일치하는 후보가 있는 경우 - 검증에서 확인된 만큼 기본 신뢰도를 준다
    score = 0.5
    score += 0.2 * int(match_info.get("disaster_type_match", 0))   # 0.1 -> 0.2
    score += 0.2 * match_info.get("text_similarity", 0.0)            # 0.3 -> 0.2 (검증에서 변별력 낮음 확인)
    score += 0.1 * (1 - min(match_info.get("time_diff_hours", 24) / 24, 1))  # 0.2 -> 0.1 (미검증)

    return {"score": round(min(score, 1.0), 3), "note": None}


def final_alert_score(disaster_info: dict, raw_url: str, model,
                       w1: float = 0.6, w2: float = 0.4) -> dict:

    d_result = disaster_reliability_score(disaster_info)
    d_score = d_result["score"]
    u_score = url_risk_score_model(raw_url, model) if raw_url else 0.0

    final = w1 * d_score + w2 * (1 - u_score)

    return {
        "disaster_reliability_score": d_score,
        "disaster_reliability_note": d_result["note"],
        "url_risk_score": u_score,
        "final_score": round(final, 3),
        "decision": "approve" if final >= 0.7 else ("review" if final >= 0.4 else "reject"),
    }


# ---------------------------------------------------------------------------
# 실행 예시 / 검증
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    model = load_url_model("/content/drive/MyDrive/REDRED/rf_url_model.joblib")

    test_urls = [
        "https://www.safekorea.go.kr/notice/123",   # 공식 도메인
        "http://vo.la/Vo27SU",                       # 공식기관이 쓰는 단축 URL(화이트리스트 대상은 아니고, 모델이 단독으로 어떻게 보는지 확인용)
        "http://192.168.1.1/login.php",              # IP + 의심 키워드
        "http://korea1l9-go-kr.verify-account.tk/secure/login",  # 타이포스쿼팅 + 의심 키워드 + 위험 TLD
    ]

    for url in test_urls:
        feats = extract_url_features(url)
        score = url_risk_score_model(url, model)
        print(f"URL: {url}")
        print(f"  피처: {feats}")
        print(f"  모델 위험도: {score:.4f}\n")

    print("\n=== final_alert_score 시나리오 검증 ===")
    scenarios = [
        ("완전 매칭 + 안전 URL", {"region_match": 1, "disaster_type_match": 1, "text_similarity": 0.6, "time_diff_hours": 1}, "https://www.safekorea.go.kr/notice/123"),
        ("정보부족 + 안전 URL", {"region_match": 0}, "https://www.safekorea.go.kr/notice/123"),
        ("정보부족 + 위험 URL", {"region_match": 0}, "http://192.168.1.1/login.php"),
        ("지역만 일치 + 안전 URL", {"region_match": 1, "disaster_type_match": 0, "text_similarity": 0.3, "time_diff_hours": 12}, "http://vo.la/Vo27SU"),
    ]
    for name, disaster_info, url in scenarios:
        result = final_alert_score(disaster_info, url, model)
        print(f"[{name}] {result}")


URL: https://www.safekorea.go.kr/notice/123
  피처: {'url_length': 38, 'domain_length': 19, 'dot_count': 3, 'slash_count': 4, 'hyphen_count': 0, 'digit_count': 3, 'contains_at': 0, 'has_ip': 0, 'https_flag': 1, 'shortener': 0, 'suspicious_word_count': 0}
  모델 위험도: 0.0000

URL: http://vo.la/Vo27SU
  피처: {'url_length': 19, 'domain_length': 5, 'dot_count': 1, 'slash_count': 3, 'hyphen_count': 0, 'digit_count': 2, 'contains_at': 0, 'has_ip': 0, 'https_flag': 0, 'shortener': 1, 'suspicious_word_count': 0}
  모델 위험도: 0.0000

URL: http://192.168.1.1/login.php
  피처: {'url_length': 28, 'domain_length': 11, 'dot_count': 4, 'slash_count': 3, 'hyphen_count': 0, 'digit_count': 8, 'contains_at': 0, 'has_ip': 1, 'https_flag': 0, 'shortener': 0, 'suspicious_word_count': 1}
  모델 위험도: 1.0000

URL: http://korea1l9-go-kr.verify-account.tk/secure/login
  피처: {'url_length': 52, 'domain_length': 32, 'dot_count': 2, 'slash_count': 4, 'hyphen_count': 3, 'digit_count': 2, 'contains_at': 0, 'has_ip': 0, 'https_flag